# SciPy 学习笔记（入门速通）

> 目标：了解 SciPy 的子模块结构，掌握优化、积分、插值、信号、稀疏矩阵、统计与空间算法的核心函数。

SciPy（Scientific Python）构建在 NumPy 之上，是 Python 科学计算的核心库。NumPy 提供基础数组和逐元素运算，SciPy 则提供**更高层的科学算法**：求根、优化、数值积分、插值、滤波、稀疏矩阵、统计分布、空间搜索等。

**环境准备**
```bash
pip install scipy
```

In [1]:
import scipy
import numpy as np
print("SciPy version:", scipy.__version__)

# 查看所有子模块
import pkgutil
submodules = [m.name for m in pkgutil.iter_modules(scipy.__path__) if not m.name.startswith('_')]
print("子模块:", submodules)

SciPy version: 1.16.1
子模块: ['cluster', 'conftest', 'constants', 'datasets', 'differentiate', 'fft', 'fftpack', 'integrate', 'interpolate', 'io', 'linalg', 'misc', 'ndimage', 'odr', 'optimize', 'signal', 'sparse', 'spatial', 'special', 'stats', 'version']


## 1. 库结构与子模块概览

SciPy 按功能领域拆分为多个子模块，按需导入。了解每个模块的职责，是用好 SciPy 的第一步。

| 子模块 | 功能 | 典型函数 |
|---|---|---|
| `scipy.constants` | 物理/数学常数 | `pi`、`c`、`golden` |
| `scipy.special` | 特殊数学函数 | `gamma`、`erf`、`bessel` |
| `scipy.optimize` | 优化与求根 | `minimize`、`root`、`curve_fit` |
| `scipy.integrate` | 数值积分 | `quad`、`solve_ivp` |
| `scipy.interpolate` | 插值 | `interp1d`、`CubicSpline` |
| `scipy.signal` | 信号处理 | `butter`、`filtfilt`、`find_peaks` |
| `scipy.linalg` | 线性代数 | `solve`、`eig`、`svd` |
| `scipy.sparse` | 稀疏矩阵 | `csr_matrix`、`eye` |
| `scipy.stats` | 统计 | `norm`、`ttest_ind`、`pearsonr` |
| `scipy.spatial` | 空间算法 | `KDTree`、`ConvexHull` |

**导入约定**：`from scipy import optimize, integrate, stats` 最常见。子模块需显式导入，`import scipy` 本身不会自动加载所有子模块。

In [2]:
from scipy import constants, special
print("圆周率:", constants.pi)
print("光速:", constants.c, "m/s")
print("黄金比例:", constants.golden)
print("伽马函数 Γ(5):", special.gamma(5))       # 4! = 24
print("误差函数 erf(1):", special.erf(1))

圆周率: 3.141592653589793
光速: 299792458.0 m/s
黄金比例: 1.618033988749895
伽马函数 Γ(5): 24.0
误差函数 erf(1): 0.8427007929497148


## 2. 优化与求根（scipy.optimize）

用于**最优化**（找函数极值）和**方程求根**。

| 函数 | 用途 | 关键参数 |
|---|---|---|
| `minimize(fun, x0, method)` | 多维函数最小化 | `method` 可选 `'BFGS'`、`'Nelder-Mead'`、`'L-BFGS-B'` 等 |
| `minimize_scalar(fun)` | 一维函数最小化 | 返回 `x`、`fun` |
| `root(fun, x0)` | 求方程 `fun(x)=0` | 返回 `x` 数组 |
| `curve_fit(f, xdata, ydata, p0)` | 非线性最小二乘拟合 | 返回参数和协方差矩阵 |

**返回值**：优化函数返回 `OptimizeResult` 对象，关键字段是 `x`（最优点）、`fun`（最优值）、`success`（是否收敛）。

**易错点**：`minimize` 需要**初始猜测** `x0`，不同初值可能收敛到不同局部极小值；多变量函数要接受数组参数。

In [3]:
from scipy import optimize
import numpy as np

# 一维最小化：f(x) = (x-3)^2 + 1
res = optimize.minimize_scalar(lambda x: (x - 3)**2 + 1)
print("最小值点:", res.x, " 最小值:", res.fun)

# 多维最小化：Rosenbrock 函数
def rosen(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

res = optimize.minimize(rosen, x0=[0, 0], method='BFGS')
print("Rosenbrock 最优:", res.x.round(3), " success:", res.success)

# 求根：x^2 - 4 = 0
root = optimize.root(lambda x: x**2 - 4, x0=1.0)
print("求根:", root.x)

# 曲线拟合：y = a*exp(-b*x)
xdata = np.linspace(0, 4, 20)
ydata = 2.5 * np.exp(-1.3 * xdata) + 0.05 * np.random.default_rng(0).normal(size=20)
popt, pcov = optimize.curve_fit(lambda x, a, b: a * np.exp(-b * x), xdata, ydata)
print("拟合参数 a,b:", popt.round(3))

最小值点: 3.0  最小值: 1.0
Rosenbrock 最优: [1. 1.]  success: True
求根: [2.]
拟合参数 a,b: [2.518 1.314]


## 3. 数值积分（scipy.integrate）

| 函数 | 用途 | 说明 |
|---|---|---|
| `quad(f, a, b)` | 一重定积分 | 返回 `(积分值, 误差估计)` |
| `dblquad(f, a, b, gfun, hfun)` | 二重积分 | 内层上下限可为函数 |
| `solve_ivp(f, t_span, y0, method)` | 常微分方程初值问题 | **推荐**，替代旧 `odeint` |
| `simpson(y, x)` / `trapezoid(y, x)` | 离散数据积分 | 已知采样点而非函数时使用 |

**`quad` 的用法**：被积函数必须接受标量或可广播数组，返回标量。积分无解析解时，用 `quad` 得到高精度数值结果。

**`solve_ivp` 与 `odeint`**：新代码用 `solve_ivp`，支持更多求解器和事件检测；`odeint` 为兼容旧代码保留。

In [4]:
from scipy import integrate
import numpy as np

# 定积分 ∫_0^π sin(x) dx = 2
val, err = integrate.quad(np.sin, 0, np.pi)
print(f"∫sin(x) (0,π) = {val:.6f}, 误差估计 {err:.2e}")

# 二重积分 ∫∫ (x*y) 在 [0,1]x[0,2]
val2, err2 = integrate.dblquad(lambda y, x: x*y, 0, 1, 0, 2)
print(f"二重积分 = {val2:.4f}")

# 常微分方程：dy/dt = -2*y, y(0)=1
sol = integrate.solve_ivp(lambda t, y: -2*y, [0, 2], [1.0], dense_output=True)
print("t=2 时 y:", sol.y[0, -1].round(4))

# 离散数据积分
x = np.linspace(0, np.pi, 100)
y = np.sin(x)
print("simpson:", integrate.simpson(y, x).round(4))

∫sin(x) (0,π) = 2.000000, 误差估计 2.22e-14
二重积分 = 1.0000
t=2 时 y: 0.0184
simpson: 2.0


## 4. 插值（scipy.interpolate）

| 函数/类 | 用途 | 特点 |
|---|---|---|
| `interp1d(x, y, kind)` | 一维插值 | 简单，`kind` 可选 `'linear'`、`'cubic'`、`'nearest'` |
| `CubicSpline(x, y)` | 三次样条 | 平滑连续，适合曲线拟合 |
| `UnivariateSpline(x, y, s)` | 平滑样条 | `s` 控制平滑度，可处理噪声数据 |
| `griddata(points, values, xi)` | 散点插值 | 非结构化二维数据 |

**`interp1d` vs `CubicSpline`**：`interp1d` 是快速通用接口，`CubicSpline` 专门做三次样条，数学上更清晰。新版 SciPy 建议新代码优先使用 `CubicSpline` 或 `make_interp_spline`。

**易错点**：插值**不能外推**——传入超出 `x` 范围的值会报错或产生不可靠结果，除非用 `fill_value='extrapolate'`（有风险）。

In [ ]:
from scipy import interpolate
import numpy as np

# 构造采样点
x = np.linspace(0, 10, 10)
y = np.sin(x) + 0.1 * np.random.default_rng(0).normal(size=10)

# 线性插值
f_linear = interpolate.interp1d(x, y, kind='linear', fill_value='extrapolate')
print("线性插值 x=5.5:", f_linear(5.5).round(4))

# 三次样条
cs = interpolate.CubicSpline(x, y)
print("三次样条 x=5.5:", cs(5.5).round(4))

# 平滑样条（处理噪声）
us = interpolate.UnivariateSpline(x, y, s=0.5)
print("平滑样条 x=5.5:", us(5.5).round(4))

# 二维散点插值
pts = np.random.default_rng(0).random((20, 2)) * 10
vals = np.sin(pts[:, 0]) + np.cos(pts[:, 1])
grid_x, grid_y = np.mgrid[0:10:10j, 0:10:10j]
grid_z = interpolate.griddata(pts, vals, (grid_x, grid_y), method='cubic')
print("网格插值形状:", grid_z.shape)

## 5. 信号处理（scipy.signal）

| 函数 | 用途 |
|---|---|
| `butter(N, Wn, btype)` | 设计 Butterworth 滤波器 |
| `filtfilt(b, a, x)` | 零相位滤波（前后各滤一次，无相移） |
| `find_peaks(x, height)` | 检测峰值位置 |
| `convolve(a, v)` / `fftconvolve` | 卷积 |
| `periodogram(x, fs)` | 功率谱估计 |

**`Wn` 的含义**：对于数字滤波器，`Wn` 是**归一化截止频率**，范围 0~1，1 对应奈奎斯特频率（fs/2）。例如采样率 1000 Hz、截止 50 Hz，则 `Wn = 50 / (1000/2) = 0.1`。

**`filtfilt` vs `lfilter`**：`lfilter` 单向滤波会引入相移，`filtfilt` 前后各滤一次抵消相移，但要求信号足够长。

In [ ]:
from scipy import signal
import numpy as np

# 构造含噪声信号：5Hz 正弦 + 高频噪声
fs = 500
t = np.linspace(0, 1, fs, endpoint=False)
clean = np.sin(2 * np.pi * 5 * t)
noisy = clean + 0.5 * np.random.default_rng(0).normal(size=fs)

# 低通滤波：截止 20Hz
b, a = signal.butter(N=4, Wn=20 / (fs / 2), btype='low')
filtered = signal.filtfilt(b, a, noisy)
print("滤波前后噪声水平:", noisy.std().round(3), "->", filtered.std().round(3))

# 峰值检测
peaks, props = signal.find_peaks(clean, height=0.9)
print("检测到峰值数:", len(peaks))

# 功率谱
freqs, psd = signal.periodogram(clean, fs)
dominant = freqs[np.argmax(psd)]
print("主频:", dominant, "Hz")

## 6. 线性代数与稀疏矩阵

`scipy.linalg` 提供比 NumPy 更丰富的线性代数函数，`scipy.sparse` 专门处理稀疏矩阵。

**scipy.linalg**

| 函数 | 用途 |
|---|---|
| `solve(A, b)` | 解线性方程组 |
| `lu(A)` / `qr(A)` / `cholesky(A)` | 矩阵分解 |
| `eig(A)` / `svd(A)` | 特征值 / 奇异值分解 |

**scipy.sparse**

| 类 | 存储格式 | 适用场景 |
|---|---|---|
| `csr_matrix` | 行压缩 | 快速矩阵乘法、行切片 |
| `csc_matrix` | 列压缩 | 列切片、求逆 |
| `coo_matrix` | 坐标列表 | 快速构造稀疏矩阵 |

**稀疏矩阵的价值**：当矩阵中非零元素占比很小时，稀疏存储只保存非零元素和索引，内存占用可降低几个数量级。

In [ ]:
from scipy import linalg, sparse
import numpy as np

# 线性代数
A = np.array([[3., 1.], [1., 2.]])
b = np.array([9., 8.])
x = linalg.solve(A, b)
print("solve:", x)
lu, piv = linalg.lu_factor(A)
print("LU 分解完成")

# 稀疏矩阵
row = np.array([0, 0, 1, 2])
col = np.array([0, 2, 2, 0])
data = np.array([1.0, 2.0, 3.0, 4.0])
coo = sparse.coo_matrix((data, (row, col)), shape=(3, 3))
csr = coo.tocsr()
print("稀疏矩阵:\n", csr.toarray())
print("非零元素数:", csr.nnz)
print("稀疏矩阵乘向量:", csr @ np.array([1, 2, 3]))

## 7. 统计（scipy.stats）

`scipy.stats` 提供概率分布、统计检验和描述性统计。

**概率分布对象**：`stats.norm`、`stats.t`、`stats.uniform` 等，常用方法：
- `pdf(x)` / `cdf(x)`：概率密度 / 累积分布
- `rvs(size)`：随机采样
- `fit(data)`：从数据估计分布参数

**常用检验**

| 函数 | 用途 |
|---|---|
| `ttest_ind(a, b)` | 两独立样本 t 检验 |
| `pearsonr(x, y)` | 皮尔逊相关系数 + p 值 |
| `chi2_contingency(table)` | 卡方独立性检验 |

**`fit` 返回什么**：返回分布形状参数 + 位置参数 + 尺度参数的元组，具体顺序因分布而异。

In [ ]:
from scipy import stats
import numpy as np

# 正态分布
print("N(0,1) 在 1.96 的 CDF:", stats.norm.cdf(1.96).round(4))
print("N(0,1) 的 95% 分位数:", stats.norm.ppf(0.95).round(4))

# 采样与拟合
rng = np.random.default_rng(0)
data = rng.normal(loc=5, scale=2, size=500)
mu, sigma = stats.norm.fit(data)
print(f"拟合参数: mu={mu:.2f}, sigma={sigma:.2f}")

# t 检验
a = rng.normal(0, 1, 100)
b = rng.normal(0.5, 1, 100)
t_stat, p_val = stats.ttest_ind(a, b)
print(f"t={t_stat:.3f}, p={p_val:.4f}")

# 相关系数
r, p = stats.pearsonr(a, b)
print(f"Pearson r={r:.3f}, p={p:.3f}")

## 8. 空间算法（scipy.spatial）

| 类 | 用途 |
|---|---|
| `KDTree(points)` | 快速最近邻查询 |
| `distance_matrix(A, B)` | 成对距离矩阵 |
| `ConvexHull(points)` | 凸包 |
| `Delaunay(points)` | 三角剖分 |

**KDTree 的优势**：对大规模点集做最近邻搜索，复杂度从 O(N) 降到 O(log N)，是空间索引的核心工具。

In [ ]:
from scipy import spatial
import numpy as np

rng = np.random.default_rng(0)
points = rng.random((100, 2)) * 10
tree = spatial.KDTree(points)

# 查询距离目标最近的 3 个点
target = [5.0, 5.0]
dist, idx = tree.query(target, k=3)
print("最近 3 点距离:", dist.round(3))
print("对应坐标:\n", points[idx])

# 凸包
hull = spatial.ConvexHull(points)
print("凸包顶点数:", len(hull.vertices))
print("凸包面积:", hull.volume.round(3))

## 9. 综合练习

1. 用 `optimize.curve_fit` 拟合 `y = a*sin(b*x) + c` 到含噪数据，打印参数。
2. 用 `integrate.quad` 计算 `∫₀^1 e^{-x²} dx`，并与 `scipy.special.erf` 的解析结果对比。
3. 用 `signal.butter` + `filtfilt` 对含噪信号做带通滤波（10~30 Hz），采样率 200 Hz。
4. 用 `sparse.random` 生成 1000×1000 的稀疏矩阵，比较稀疏存储与稠密存储的内存占用。

**参考答案**

In [ ]:
import numpy as np
from scipy import optimize, integrate, special, signal, sparse

# 1. 曲线拟合
rng = np.random.default_rng(1)
x = np.linspace(0, 10, 60)
y = 2.0*np.sin(0.8*x) + 0.5 + 0.1*rng.normal(size=60)
popt, _ = optimize.curve_fit(lambda x,a,b,c: a*np.sin(b*x)+c, x, y, p0=[1,1,0])
print("1. 拟合 a,b,c:", popt.round(3))

# 2. 高斯积分
val, _ = integrate.quad(lambda x: np.exp(-x**2), 0, 1)
exact = np.sqrt(np.pi)/2 * special.erf(1)
print(f"2. quad={val:.8f}, erf={exact:.8f}, 差={abs(val-exact):.2e}")

# 3. 带通滤波
fs = 200
t = np.linspace(0, 2, 2*fs, endpoint=False)
sig = np.sin(2*np.pi*20*t) + np.sin(2*np.pi*60*t) + 0.3*rng.normal(size=len(t))
b, a = signal.butter(4, [10/(fs/2), 30/(fs/2)], btype='band')
filt = signal.filtfilt(b, a, sig)
print(f"3. 滤波后 20Hz 成分保留，60Hz 衰减，std {sig.std():.3f} -> {filt.std():.3f}")

# 4. 稀疏 vs 稠密内存
sp = sparse.random(1000, 1000, density=0.01, format='csr', random_state=0)
dense_bytes = 1000*1000*8
sparse_bytes = sp.data.nbytes + sp.indices.nbytes + sp.indptr.nbytes
print(f"4. 稠密 ~{dense_bytes/1024**2:.1f} MB, 稀疏 ~{sparse_bytes/1024**2:.2f} MB")

## 常见坑点总结

- **子模块需显式导入**：`import scipy` 不会自动加载 `scipy.optimize`，要写 `from scipy import optimize`。
- **`quad` 返回元组**：`val, err = integrate.quad(...)`，不是直接返回数值。
- **`minimize` 需要初始值**：`x0` 选得不好会陷入局部极小，多变量函数必须接受数组。
- **滤波器的 `Wn` 是归一化频率**：除以奈奎斯特频率（fs/2），不是直接填 Hz。
- **插值不能随意外推**：超出范围的值不可靠，`fill_value='extrapolate'` 需谨慎。
- **稀疏矩阵别转稠密**：`toarray()` 在超大矩阵上会爆内存，尽量全程保持稀疏格式。
- **`odeint` 是旧接口**：新代码用 `solve_ivp`，支持事件检测和更多求解器。